In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas_gbq

query = """
SELECT *
FROM `amplified-alpha-463610-c8.ecommerce_ml.session_features`
"""

df = pandas_gbq.read_gbq(
    query,
    project_id="amplified-alpha-463610-c8",
    dialect="standard"
)

df.head()

Downloading: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████|


,user_pseudo_id,session_id,session_date,total_events_5min,page_views_5min,product_views_5min,product_clicks_5min,add_to_cart_5min,begin_checkout_5min,search_events_5min,is_new_user,device_category,operating_system,mobile_brand,mobile_model,traffic_source,traffic_medium,target
0,56162479.7328415035,3120510756,2020-11-21,13,4,2,0,0,0,0,0,mobile,Android,Samsung,<Other>,(data deleted),(data deleted),0
1,38649160.6784990719,5087740138,2020-11-30,2,0,0,0,0,0,0,0,mobile,Android,Samsung,<Other>,(data deleted),(data deleted),0
2,34497430.3741988669,6112241130,2020-12-18,22,9,1,1,0,0,0,0,mobile,Web,Samsung,<Other>,<Other>,(data deleted),0
3,54536524.4339644387,2096827012,2021-01-28,5,2,0,0,0,0,0,0,mobile,Web,<Other>,<Other>,(data deleted),(data deleted),0
4,16851116.6419517132,7459325817,2020-12-07,11,3,3,0,0,0,0,0,mobile,Web,<Other>,<Other>,(data deleted),(data deleted),0


In [3]:
data = df.copy()

In [4]:
data["session_date"] = pd.to_datetime(data["session_date"])

In [5]:
data["session_date"].min(), data["session_date"].max()

(Timestamp('2020-11-01 00:00:00'), Timestamp('2021-01-31 00:00:00'))

In [6]:
data["has_cart"] = (
    data["add_to_cart_5min"] > 0
).astype(int)

data["has_checkout"] = (
    data["begin_checkout_5min"] > 0
).astype(int)

data["has_product_view"] = (
    data["product_views_5min"] > 0
).astype(int)

data["has_product_click"] = (
    data["product_clicks_5min"] > 0
).astype(int)

In [7]:
data["product_views_per_page"] = (
    data["product_views_5min"] /
    data["page_views_5min"].replace(0, np.nan)
)

data["click_per_product_view"] = (
    data["product_clicks_5min"] /
    data["product_views_5min"].replace(0, np.nan)
)

data["cart_per_product_view"] = (
    data["add_to_cart_5min"] /
    data["product_views_5min"].replace(0, np.nan)
)

In [8]:
ratio_features = [
    "product_views_per_page",
    "click_per_product_view",
    "cart_per_product_view"
]

data[ratio_features] = (
    data[ratio_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [9]:
data["is_returning_user"] = (
    1 - data["is_new_user"]
)

In [10]:
data["day_of_week"] = (
    data["session_date"].dt.dayofweek
)

data["is_weekend"] = (
    data["day_of_week"].isin([5, 6])
).astype(int)

In [11]:
categorical_cols = [
    "device_category",
    "operating_system",
    "mobile_brand",
    "mobile_model",
    "traffic_source",
    "traffic_medium"
]

for col in categorical_cols:
    print(
        "\n",
        col,
        data[col].nunique(),
        data[col].isna().sum()
    )


 device_category 3 0

 operating_system 6 0

 mobile_brand 8 0

 mobile_model 10 0

 traffic_source 5 0

 traffic_medium 6 0


In [12]:
for col in categorical_cols:
    data[col] = data[col].fillna("unknown")

In [13]:
drop_cols = [
    "user_pseudo_id",
    "session_id"
]

In [14]:
train = data[
    data["session_date"] <= "2020-12-15"
].copy()

validation = data[
    (data["session_date"] >= "2020-12-16") &
    (data["session_date"] <= "2021-01-07")
].copy()

test = data[
    data["session_date"] >= "2021-01-08"
].copy()

In [15]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (184148, 28)
Validation: (83529, 28)
Test: (91926, 28)


In [16]:
numeric_features = [
    "total_events_5min",
    "page_views_5min",
    "product_views_5min",
    "product_clicks_5min",
    "add_to_cart_5min",
    "begin_checkout_5min",
    "search_events_5min",
    "is_returning_user",
    "has_cart",
    "has_checkout",
    "has_product_view",
    "has_product_click",
    "product_views_per_page",
    "click_per_product_view",
    "cart_per_product_view",
    "is_weekend"
]

categorical_features = [
    "device_category",
    "operating_system",
    "traffic_source",
    "traffic_medium"
]

In [17]:
feature_columns = (
    numeric_features +
    categorical_features
)

X_train = train[feature_columns]
y_train = train["target"]

X_val = validation[feature_columns]
y_val = validation["target"]

X_test = test[feature_columns]
y_test = test["target"]

In [18]:
X_train.shape, X_val.shape, X_test.shape

((184148, 20), (83529, 20), (91926, 20))

## Check conversion rate in each split

In [19]:
for name, dataset in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(
        name,
        "sessions =", len(dataset),
        "conversions =", dataset["target"].sum(),
        "conversion rate =",
        round(dataset["target"].mean() * 100, 3),
        "%"
    )

Train sessions = 184148 conversions = 2733 conversion rate = 1.484 %
Validation sessions = 83529 conversions = 734 conversion rate = 0.879 %
Test sessions = 91926 conversions = 855 conversion rate = 0.93 %
